Data Downlaod

In [1]:
from mp_api.client import MPRester
import pandas as pd
from tqdm import tqdm

C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pymatgen
import matminer

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
API_KEY = "SDRI2npv5evGn5E7Z1ACiKimrVQn2kJZ"

In [6]:
# from mp_api.client import MPRester
# import pandas as pd
# from tqdm import tqdm

# API_KEY = "SDRI2npv5evGn5E7Z1ACiKimrVQn2kJZ"

# data = []

# with MPRester(API_KEY) as mpr:
    
#     docs = mpr.materials.summary.search(
#     nelements=(3, 5),
#     fields=[
#         "material_id",
#         "formula_pretty",
#         "band_gap",
#         "structure",   # ⭐ IMPORTANT
#         "density",
#         "volume"
#     ]
#     )
    
# for i, d in enumerate(tqdm(docs)):
#     if i >= 20000:
#         break
    
#     structure = d.structure
    
#     data.append({
#         "material_id": d.material_id,
#         "formula": d.formula_pretty,
#         "bandgap": d.band_gap,
        
#         # Structure features
#         "a": structure.lattice.a,
#         "b": structure.lattice.b,
#         "c": structure.lattice.c,
        
#         "volume": d.volume,
#         "density": d.density
#         })

# df = pd.DataFrame(data)

# print("Downloaded materials:", len(df))
# df.head()
#######################################

from mp_api.client import MPRester
import pandas as pd
from tqdm import tqdm

API_KEY = "SDRI2npv5evGn5E7Z1ACiKimrVQn2kJZ"

def download_perovskites():
    all_data = []
    # List of common perovskite formula patterns to search
    patterns = ["**O3", "**I3", "**Br3", "**Cl3", "**F3", "*2*6"]
    
    with MPRester(API_KEY) as mpr:
        for pattern in patterns:
            print(f"Fetching pattern: {pattern}...")
            try:
                # Using 'num_elements' as requested by the warning
                docs = mpr.materials.summary.search(
                    formula=pattern, 
                    num_elements=(3, 6), # Broadened to include hybrids
                    is_stable=False, 
                    fields=["material_id", "formula_pretty", "band_gap", "structure", "density", "volume"]
                )
                
                for d in tqdm(docs):
                    s = d.structure
                    all_data.append({
                        "material_id": str(d.material_id),
                        "formula": d.formula_pretty,
                        "bandgap": d.band_gap,
                        "a": s.lattice.a,
                        "b": s.lattice.b,
                        "c": s.lattice.c,
                        "volume": d.volume,
                        "density": d.density,
                        "structure": s 
                    })
            except Exception as e:
                print(f"Error fetching {pattern}: {e}")
                continue

    return pd.DataFrame(all_data)

# Run and save
df = download_perovskites()
if len(df) > 0:
    df.to_csv("perovskite_base_data.csv")
    print(f"Success! Saved {len(df)} entries.")
else:
    print("Still 0 entries. Please check if your API Key is active on the Materials Project dashboard.")

Fetching pattern: **O3...


100%|██████████| 2230/2230 [00:00<00:00, 86680.86it/s]


Fetching pattern: **I3...


100%|██████████| 63/63 [00:00<00:00, 57232.22it/s]


Fetching pattern: **Br3...


...

Fetching pattern: **F3...


100%|██████████| 311/311 [00:00<00:00, 106146.03it/s]


Fetching pattern: *2*6...


Retrieving SummaryDoc documents: 0it [00:00, ?it/s]
0it [00:00, ?it/s]


Success! Saved 2863 entries.


In [7]:
# Assuming 'df' is your downloaded dataframe
print(f"Before cleaning: {len(df)}")

# 1. Remove exact duplicate Material IDs (Safety check)
df = df.drop_duplicates(subset=['material_id'])

# 2. Sort by bandgap or a stability metric if you have it
# For now, let's keep all unique material_ids because even if the formula
# is the same, the material_id is unique for each specific structure.
df = df.sort_values(by='formula')

print(f"After cleaning: {len(df)}")

Before cleaning: 2863
After cleaning: 2863


In [2]:
from mp_api.client import MPRester
import pandas as pd
from tqdm import tqdm

# Note: Keep your API key private in final thesis submissions!
API_KEY = "SDRI2npv5evGn5E7Z1ACiKimrVQn2kJZ"

data = []

with MPRester(API_KEY) as mpr:
    print("Querying Materials Project for Perovskites...")
    # We search for ABX3 patterns. 
    # 'is_stable=True' is recommended so your ML learns from real-world physics,
    # but if you need exactly 20k, we can set it to False to include theoretical ones.
    docs = mpr.materials.summary.search(
        formula="*3", 
        nelements=(3, 5),
        is_stable=True, 
        fields=[
            "material_id",
            "formula_pretty",
            "band_gap",
            "structure",
            "density",
            "volume"
        ]
    )

print(f"Total materials found: {len(docs)}")

# Processing the first 20,000 results
for d in tqdm(docs[:20000]):
    try:
        struct = d.structure
        
        data.append({
            "material_id": str(d.material_id),
            "formula": d.formula_pretty,
            "bandgap": d.band_gap,
            
            # Lattice Constants
            "a": struct.lattice.a,
            "b": struct.lattice.b,
            "c": struct.lattice.c,
            
            # Lattice Angles (Crucial for Perovskite distortions)
            "alpha": struct.lattice.alpha,
            "beta": struct.lattice.beta,
            "gamma": struct.lattice.gamma,
            
            "volume": d.volume,
            "density": d.density,
            
            # Storing the structure object temporarily helps MatMiner later
            "structure_obj": struct 
        })
    except Exception as e:
        continue

df = pd.DataFrame(data)

# Save raw data immediately so you don't have to keep calling the API
df.to_pickle("perovskite_raw_data.pkl") 
print(f"Successfully processed {len(df)} materials.")
df.head()

Querying Materials Project for Perovskites...


C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\mp_api\client\routes\materials\summary.py:278: MPRestWarning: You have specified fields used by `_search` that can be understood by `search`
   nelements
To ensure long term support, please use their `search` equivalents:
   num_elements
Please see the documentation:
    `search`: https://materialsproject.github.io/api/_autosummary/mp_api.client.routes.materials.summary.SummaryRester.html#mp_api.client.routes.materials.summary.SummaryRester.search
   `_search`: https://api.materialsproject.org/redoc#tag/Materials-Summary/operation/search_materials_summary__get
  warnings.warn(
Retrieving SummaryDoc documents: 0it [00:00, ?it/s]


Total materials found: 0


0it [00:00, ?it/s]


Successfully processed 0 materials.


""


In [11]:
print("Total perovskite materials:", len(df))

print("\nColumns:")
print(df.columns)

print("\nSample data:")
df.head()

Total perovskite materials: 1000

Columns:
Index(['material_id', 'formula', 'bandgap', 'elements', 'num_elements'], dtype='object')

Sample data:


,material_id,formula,bandgap,elements,num_elements
0,mp-1183115,AcAlO3,4.1024,"[Ac, Al, O]",3
1,mp-1183052,AcBO3,0.8071,"[Ac, B, O]",3
2,mp-30274,AcBrO,4.2410,"[Ac, Br, O]",3
3,mp-30273,AcClO,4.4451,"[Ac, Cl, O]",3
4,mp-866101,AcCrO3,2.0031,"[Ac, Cr, O]",3


In [12]:
df.to_csv("perovskite_dataset_large.csv", index=False)

print("Perovskite dataset saved successfully")

Perovskite dataset saved successfully


In [1]:
from pymatgen.core import Composition

def is_perovskite(formula):
    try:
        comp = Composition(formula)
        el_amt = comp.get_el_amt_dict()
        
        # Must have exactly 3 elements
        if len(el_amt) != 3:
            return False
        
        values = list(el_amt.values())
        values.sort()
        
        # Check ratio ~1:1:3
        if values[0] == 1 and values[1] == 1 and values[2] == 3:
            return True
        
        return False
    
    except:
        return False


# Apply filter
df["is_perovskite"] = df["formula"].apply(is_perovskite)

perov_df = df[df["is_perovskite"] == True]

print("Perovskite materials found:", len(perov_df))
perov_df.head()

NameError: name 'df' is not defined

In [5]:
print("Total perovskite materials:", len(perov_df))

print("\nColumns:")
print(perov_df.columns)

print("\nSample data:")
perov_df.head()

Total perovskite materials: 97

Columns:
Index(['material_id', 'formula', 'bandgap', 'elements', 'num_elements',
       'is_perovskite'],
      dtype='object')

Sample data:


,material_id,formula,bandgap,elements,num_elements,is_perovskite
0,mp-1183115,AcAlO3,4.1024,"[Ac, Al, O]",3,True
1,mp-1183052,AcBO3,0.8071,"[Ac, B, O]",3,True
4,mp-866101,AcCrO3,2.0031,"[Ac, Cr, O]",3,True
5,mp-861502,AcFeO3,0.9888,"[Ac, Fe, O]",3,True
6,mp-1183053,AcGaO3,2.8959,"[Ac, Ga, O]",3,True


In [7]:
print("\nMissing values:")
print(perov_df.isnull().sum())


Missing values:
material_id      0
formula          0
bandgap          0
elements         0
num_elements     0
is_perovskite    0
dtype: int64


In [8]:
all_elements = set()

for elems in perov_df["elements"]:
    all_elements.update(elems)

print("Unique elements in dataset:")
print(all_elements)

Unique elements in dataset:
{Element As, Element Se, Element Br, Element B, Element C, Element N, Element O, Element F, Element Mo, Element W, Element Al, Element Rh, Element Si, Element Ag, Element P, Element Au, Element S, Element Cl, Element In, Element Tl, Element Sb, Element Bi, Element Te, Element Sc, Element I, Element V, Element Cr, Element Ac, Element Fe, Element Co, Element Cu, Element Ga}


In [9]:
perov_df.to_csv("perovskite_dataset_small.csv", index=False)

print("Perovskite dataset saved successfully")

Perovskite dataset saved successfully
